# Process images

Runs `mechcell.ProcessImage` on selected nuclei and shows each result in
a napari viewer to inspect the segmentation of the nucleus, the HP1α condensates and the
pillars. This notebook was used to process all images.

The viewer needs the `viewer` extra: `uv sync --extra notebook --extra viewer`.

In [ ]:
import json
import os
import re
from pathlib import Path

import pandas as pd

from mechcell import ProcessImage

pd.set_option("display.max_columns", None)

## Settings

In [ ]:
# The data/ folder
ROOT = Path("../../data")
# Example metadata JSON files of the session's nuclei
META_FILES = "REP1/24hours/1600nm/260422_1600nm_4umpitch_24h_hela_experiments_meta/*.json"
# Example coordinate folder of the replicate
COORDINATE_DIR = Path("coordinate_files_REP1")
# Example session table
OUTPUT_TABLE = Path("process_image_results/260422_REP1_24h_1600nm_4umpitch_all_data_nuclei_ilastik.csv")
# Open each result in napari
SHOW_VIEWER = True

## Process the nuclei

In [ ]:
os.chdir(ROOT)
meta_files = sorted(Path(".").glob(META_FILES))
print(f"{len(meta_files)} metadata file(s)")

all_results = []
for nucleus_counter, meta_file in enumerate(meta_files, start=1):
    print(f"Processing: {meta_file}")
    metadata = json.loads(meta_file.read_text())

    processor = ProcessImage(
        metadata,
        data_dir=Path("."),
        coordinate_files=COORDINATE_DIR,
        meta_file=meta_file,
        show_viewer=SHOW_VIEWER,
        control=False,
    )
    processor()

    df = processor.df_img_results
    df["nucleus_id"] = nucleus_counter
    m = re.search(r"nuc(\d+)", meta_file.stem)
    df["nucleus_nr"] = int(m.group(1)) if m else None
    all_results.append(df)

# One row per condensate; nucleus columns repeat on each row
results = pd.concat(all_results, ignore_index=True)
results

## Save the session table

In [ ]:
OUTPUT_TABLE.parent.mkdir(parents=True, exist_ok=True)
results.to_csv(OUTPUT_TABLE, sep="\t")
print(f"Wrote {len(results)} rows to {ROOT / OUTPUT_TABLE}")